# Mystery & Sci-Fi — One-Click Colab Run

Six-scene preparation runner. It checks the required Gemini Developer API key first, syncs the tested branch, installs pinned dependencies, detects GPU/CPU for Whisper, prepares narration and timed captions, validates all outputs, and displays the two Google Flow text artifacts inline for copying.

- Uses the existing Google Drive project and persistent run log.
- Strict spending guard: image/video providers remain `manual_flow`; paid image and video API generation stay disabled.
- A resumed project may retain its existing plan; if its topic differs materially from `TOPIC`, the notebook logs a clear warning.
- Google Flow generation remains manual; the notebook does not spend Flow credits automatically.


In [ ]:
import os, sys, json, time, shutil, subprocess, traceback, re, difflib, html, warnings
from pathlib import Path
from google.colab import drive, userdata
from IPython.display import display, Markdown, HTML

# One-click configuration for this six-scene test.
TOPIC = "An abandoned coastal radio station receives a warning about an event that has not happened yet."
SCENE_COUNT = 6
VOICE = "en-US-ChristopherNeural"
WHISPER_MODEL = "base.en"
RESUME_PROJECT_ID = "20261120_123020_mystery_scifi_An_abandoned_coastal_radio_station_recei"
PROJECT_ID = RESUME_PROJECT_ID  # compatibility alias for the existing project layout
REPO_URL = "https://github.com/topboyasian-stack/gta6-autoclipper.git"
CODE_BRANCH = "improve/narrative-shorts-production"
ROOT = Path("/content/gta6-autoclipper")
IMAGE_PROVIDER = "manual_flow"
VIDEO_PROVIDER = "manual_flow"
ENABLE_PAID_VIDEO_API = "0"
ALLOW_PAID_IMAGE_API = "false"

# Require the Developer API key before mounting Drive, installing dependencies, or running stages.
try:
    api_key = userdata.get("GEMINI_API_KEY")
except Exception:
    api_key = None
if not api_key:
    api_key = os.environ.get("GEMINI_API_KEY", "")
api_key = str(api_key or "").strip()
if len(api_key) < 10:
    raise RuntimeError(
        "GEMINI_API_KEY is missing. Add it to Colab Secrets (recommended) or the GEMINI_API_KEY "
        "environment variable, then rerun. The run has been stopped before Drive access, package "
        "installation, or any model/API request."
    )

# Choose a CUDA Whisper path when the runtime actually exposes a GPU; use a clean CPU fallback.
WHISPER_DEVICE = "cpu"
WHISPER_COMPUTE_TYPE = "int8"
try:
    import torch
    if torch.cuda.is_available():
        WHISPER_DEVICE, WHISPER_COMPUTE_TYPE = "cuda", "float16"
except Exception:
    pass
if WHISPER_DEVICE != "cuda" and shutil.which("nvidia-smi"):
    try:
        gpu_check = subprocess.run(
            ["nvidia-smi", "-L"], capture_output=True, text=True, timeout=8, check=False
        )
        if gpu_check.returncode == 0 and gpu_check.stdout.strip():
            WHISPER_DEVICE, WHISPER_COMPUTE_TYPE = "cuda", "float16"
    except Exception:
        pass
print(
    f"Whisper runtime selected: {WHISPER_DEVICE} ({WHISPER_COMPUTE_TYPE}). "
    "CUDA initialization failures fall back to CPU int8.",
    flush=True,
)

LOG_PATH = None
try:
    if not Path("/content/drive/MyDrive").is_dir():
        print("Connecting Google Drive…", flush=True)
        drive.mount("/content/drive")

    WORKSPACE = Path("/content/drive/MyDrive/GTA6_AutoClipper")
    PROJECT_DIR = WORKSPACE / "projects" / RESUME_PROJECT_ID
    OUTPUT_DIR = PROJECT_DIR / "mystery_scifi"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    LOG_PATH = OUTPUT_DIR / "mystery_run.log"

    def note(message):
        stamp = time.strftime("%Y-%m-%d %H:%M:%S")
        line = f"[{stamp}] {message}"
        print(line, flush=True)
        with LOG_PATH.open("a", encoding="utf-8") as log:
            log.write(line + "\n")
            log.flush()

    with LOG_PATH.open("a", encoding="utf-8") as log:
        log.write("\n\n===== One-click six-scene run: " + time.strftime("%Y-%m-%d %H:%M:%S") + " =====\n")

    note("Persistent log: " + str(LOG_PATH))
    note("Project output: " + str(OUTPUT_DIR))
    note("GEMINI_API_KEY is present; the key value will not be printed.")
    note("Spend guard: image and video providers are manual_flow; paid video API is disabled.")

    def run_logged(label, command, env=None):
        note("START: " + label)
        with LOG_PATH.open("a", encoding="utf-8") as log:
            log.write("COMMAND: " + " ".join(str(part) for part in command) + "\n")
            log.flush()
            process = subprocess.Popen(
                [str(part) for part in command],
                cwd=str(ROOT) if ROOT.is_dir() else None,
                env=env,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1,
                errors="replace",
            )
            log.write("PROCESS_ID: " + str(process.pid) + "\n")
            log.flush()
            print("Process started: " + str(process.pid), flush=True)
            for line in process.stdout:
                log.write(line)
                log.flush()
                print(line, end="", flush=True)
            return_code = process.wait()
            log.write("EXIT_CODE: " + str(return_code) + "\n")
            log.flush()
        if return_code != 0:
            raise RuntimeError(f"{label} failed with exit code {return_code}. See {LOG_PATH}")
        note("DONE: " + label)

    if (ROOT / ".git").is_dir():
        run_logged(
            "Fetch narrative test branch " + CODE_BRANCH,
            ["git", "-C", str(ROOT), "fetch", "--depth=1", "origin", CODE_BRANCH],
        )
        run_logged(
            "Select the tested narrative branch commit",
            ["git", "-C", str(ROOT), "reset", "--hard", "FETCH_HEAD"],
        )
    else:
        if ROOT.exists():
            shutil.rmtree(ROOT)
        run_logged(
            "Download production code",
            ["git", "clone", "--depth", "1", "--branch", CODE_BRANCH, REPO_URL, str(ROOT)],
        )

    # The tracked requirements file is authoritative. Create a pinned studio fallback only if absent.
    requirements_path = ROOT / "requirements.txt"
    if not requirements_path.is_file():
        pinned_fallback = (
            "google-genai==2.34.0\n"
            "pydantic==2.14.4\n"
            "python-dotenv==1.2.1\n"
            "edge-tts==7.3.1\n"
            "faster-whisper==1.2.1\n"
            "av==17.1.0\n"
            "Pillow==12.3.0\n"
        )
        requirements_path.write_text(pinned_fallback, encoding="utf-8")
        note("WARNING: repository root had no requirements.txt; created a pinned studio-only fallback.")

    # Install FFmpeg/ffprobe only if either executable is missing.
    if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
        run_logged(
            "Install FFmpeg and fonts",
            ["bash", "-lc", "apt-get update -qq && apt-get install -y -qq ffmpeg fonts-dejavu-core"],
        )
    if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
        raise RuntimeError("FFmpeg setup did not produce both ffmpeg and ffprobe executables.")
    note("FFmpeg and ffprobe are available.")

    note("Install exact versions from the repository's pinned requirements.txt.")
    run_logged("Install pinned Python dependencies", [
        sys.executable, "-m", "pip", "install", "-q",
        "--disable-pip-version-check", "-r", str(requirements_path),
    ])

    # The saved plan belongs to RESUME_PROJECT_ID and may intentionally differ from the current TOPIC.
    def normalize_topic(value):
        return re.sub(r"[^a-z0-9]+", " ", str(value or "").casefold()).strip()

    def topics_differ_significantly(current, saved):
        current_norm, saved_norm = normalize_topic(current), normalize_topic(saved)
        if not current_norm or not saved_norm or current_norm == saved_norm:
            return False
        current_words, saved_words = set(current_norm.split()), set(saved_norm.split())
        union = current_words | saved_words
        overlap = len(current_words & saved_words) / max(1, len(union))
        ratio = difflib.SequenceMatcher(None, current_norm, saved_norm).ratio()
        return ratio < 0.68 and overlap < 0.50

    plan_path = OUTPUT_DIR / "episode.json"
    active_topic = TOPIC
    if plan_path.is_file():
        try:
            previous = json.loads(plan_path.read_text(encoding="utf-8"))
            previous_count = len(previous.get("scenes", []))
            previous_topic = previous.get("topic", "")
        except Exception:
            previous_count, previous_topic = -1, ""

        if previous_topic and topics_differ_significantly(TOPIC, previous_topic):
            if previous_count == SCENE_COUNT:
                message = (
                    f"WARNING: RESUME_PROJECT_ID='{RESUME_PROJECT_ID}' is overriding the current TOPIC "
                    "with the existing project's saved plan. "
                    f"Current TOPIC: {TOPIC!r}. Existing plan topic: {previous_topic!r}. "
                    "The saved plan will be preserved and reused."
                )
                warnings.warn(message, RuntimeWarning, stacklevel=2)
                note(message)
                active_topic = previous_topic
            else:
                message = (
                    f"WARNING: RESUME_PROJECT_ID='{RESUME_PROJECT_ID}' contains a topic that differs "
                    f"significantly from current TOPIC. Saved topic: {previous_topic!r}; current TOPIC: {TOPIC!r}. "
                    f"The saved plan has {previous_count} scenes, not {SCENE_COUNT}; it will be backed up "
                    "and a replacement will be generated from current TOPIC."
                )
                warnings.warn(message, RuntimeWarning, stacklevel=2)
                note(message)

        if previous_count != SCENE_COUNT:
            backup = OUTPUT_DIR / (
                "episode.backup.before_six_scene_" + time.strftime("%Y%m%d_%H%M%S") + ".json"
            )
            shutil.copy2(plan_path, backup)
            plan_path.unlink()
            active_topic = TOPIC
            note(
                f"Previous plan had {previous_count} scenes or was unreadable; "
                f"backed it up to {backup}."
            )
        else:
            note("Found a valid saved six-scene plan; the CLI will preserve it.")
            if not previous_topic:
                active_topic = TOPIC

    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    (PROJECT_DIR / "project_state.json").write_text(
        json.dumps({
            "project_name": RESUME_PROJECT_ID,
            "mode": "Mystery & Sci-Fi Automation",
            "story_title": active_topic[:160],
            "requested_topic": TOPIC,
            "mystery_output": str(OUTPUT_DIR),
            "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            "setup": "one-click-colab",
        }, indent=2),
        encoding="utf-8",
    )

    env = os.environ.copy()
    env.update({
        "PYTHONUNBUFFERED": "1",
        "GEMINI_API_KEY": api_key,
        "ALLOW_PAID_IMAGE_API": ALLOW_PAID_IMAGE_API,
        "IMAGE_PROVIDER": IMAGE_PROVIDER,
        "VIDEO_PROVIDER": VIDEO_PROVIDER,
        "ENABLE_PAID_VIDEO_API": ENABLE_PAID_VIDEO_API,
        "MYSTERY_WHISPER_DEVICE": WHISPER_DEVICE,
        "MYSTERY_WHISPER_COMPUTE_TYPE": WHISPER_COMPUTE_TYPE,
        "MYSTERY_WHISPER_MODEL": WHISPER_MODEL,
        "MYSTERY_NARRATOR_VOICE": VOICE,
        "MYSTERY_MUSIC_TRACK": str(OUTPUT_DIR / "no_music_selected.mp3"),
    })
    required_safety_values = {
        "ALLOW_PAID_IMAGE_API": "false",
        "IMAGE_PROVIDER": "manual_flow",
        "VIDEO_PROVIDER": "manual_flow",
        "ENABLE_PAID_VIDEO_API": "0",
    }
    for name, expected in required_safety_values.items():
        if env.get(name) != expected:
            raise RuntimeError(f"Spend-protection setting {name} was not enforced: expected {expected!r}.")

    command = [
        sys.executable, str(ROOT / "main.py"),
        "--stage", "prepare",
        "--topic", TOPIC,
        "--scene-count", str(SCENE_COUNT),
        "--output", str(OUTPUT_DIR),
        "--voice", VOICE,
        "--whisper-model", WHISPER_MODEL,
        "--whisper-device", WHISPER_DEVICE,
        "--max-scenes-per-flow-run", "8",
    ]
    # These options were checked against main.py's argparse parser. Paid image generation is not opted in.
    run_logged("Prepare episode, narration, timed captions, and Flow manifest", command, env=env)

    episode_path = OUTPUT_DIR / "episode.json"
    base_artifacts = [
        ("episode.json", episode_path),
        ("episode brief", OUTPUT_DIR / "episode_brief.md"),
        ("asset prompt plan", OUTPUT_DIR / "assets_generation_plan.md"),
        ("narration manifest", OUTPUT_DIR / "audio" / "narration_manifest.json"),
        ("timed captions", OUTPUT_DIR / "captions" / "subtitles.srt"),
        ("Flow JSON manifest", OUTPUT_DIR / "flow_upload_manifest.json"),
        ("Flow text manifest", OUTPUT_DIR / "flow_upload_manifest.txt"),
        ("Flow master prompt", OUTPUT_DIR / "flow_master_prompt.md"),
    ]
    missing_or_empty = []
    for label, path in base_artifacts:
        if not path.is_file():
            missing_or_empty.append(f"{label}: missing ({path})")
        elif path.stat().st_size <= 0:
            missing_or_empty.append(f"{label}: empty ({path})")
    if missing_or_empty:
        raise RuntimeError(
            "Preparation did not generate every required non-empty artifact:\n - "
            + "\n - ".join(missing_or_empty)
        )

    episode = json.loads(episode_path.read_text(encoding="utf-8"))
    actual_count = len(episode.get("scenes", []))
    if actual_count != SCENE_COUNT:
        raise RuntimeError(f"Expected {SCENE_COUNT} scenes, found {actual_count} in episode.json.")

    audio_artifacts = []
    for scene in episode["scenes"]:
        scene_id = f"scene_{int(scene['scene_number']):03d}"
        audio_artifacts.extend([
            (f"{scene_id} narration MP3", OUTPUT_DIR / "audio" / f"{scene_id}.mp3"),
            (f"{scene_id} narration WAV", OUTPUT_DIR / "audio" / f"{scene_id}.wav"),
        ])
    for label, path in audio_artifacts:
        if not path.is_file():
            missing_or_empty.append(f"{label}: missing ({path})")
        elif path.stat().st_size <= 0:
            missing_or_empty.append(f"{label}: empty ({path})")
    if missing_or_empty:
        raise RuntimeError(
            "Preparation did not generate every required non-empty artifact:\n - "
            + "\n - ".join(missing_or_empty)
        )

    note("SUCCESS: all expected plan, narration, caption, and Flow prompt artifacts exist and are non-empty.")
    note("Flow clip generation remains manual; no video API was enabled and no Flow credits were spent.")
    display(Markdown(
        "## Six-scene preparation verified\n\n"
        f"**Episode:** {episode.get('title', 'Untitled')}\n\n"
        f"**Scenes:** {actual_count}\n\n"
        f"**Resumed project:** {RESUME_PROJECT_ID}\n\n"
        f"**Google Drive project folder:** {OUTPUT_DIR}\n\n"
        f"**Persistent run log:** {LOG_PATH}\n\n"
        "**Artifact checklist:**\n"
        "- Verified non-empty episode.json\n"
        "- Verified narration manifest and MP3/WAV files for every scene\n"
        "- Verified non-empty timed SRT captions\n"
        "- Verified non-empty flow_master_prompt.md and flow_upload_manifest.txt\n\n"
        "Image generation remains prompt-only; Google Flow video generation is a separate manual step."
    ))

    def show_copy_panel(title, content, element_id):
        safe_title = html.escape(title)
        safe_content = html.escape(content)
        markup = (
            '<details open style="margin:12px 0;padding:10px;border:1px solid #bbb;border-radius:8px;">'
            f'<summary style="cursor:pointer;font-weight:600;">{safe_title}</summary>'
            f'<textarea id="{element_id}" readonly spellcheck="false" '
            'style="box-sizing:border-box;width:100%;min-height:260px;margin-top:8px;white-space:pre;'
            'font-family:monospace;font-size:12px;">'
            + safe_content +
            "</textarea>"
            f'''<button type="button" style="margin-top:8px;padding:6px 12px;" onclick="var e=this.previousElementSibling;e.focus();e.select();try{{if(navigator.clipboard&&navigator.clipboard.writeText){{navigator.clipboard.writeText(e.value);}}else{{document.execCommand('copy');}}}}catch(err){{document.execCommand('copy');}}">Copy {safe_title}</button></details>'''
        )
        display(HTML(markup))

    show_copy_panel(
        "Google Flow master prompt — copy into Flow",
        (OUTPUT_DIR / "flow_master_prompt.md").read_text(encoding="utf-8"),
        "flow-master-prompt-copy",
    )
    show_copy_panel(
        "Google Flow upload manifest — copy into Flow",
        (OUTPUT_DIR / "flow_upload_manifest.txt").read_text(encoding="utf-8"),
        "flow-upload-manifest-copy",
    )

except Exception as exc:
    error = f"FAILED: {type(exc).__name__}: {exc}"
    print("\n" + error, flush=True)
    try:
        if LOG_PATH and LOG_PATH.is_file():
            with LOG_PATH.open("a", encoding="utf-8") as log:
                log.write("\n" + error + "\n")
                log.write(traceback.format_exc() + "\n")
                log.flush()
            tail = LOG_PATH.read_text(encoding="utf-8", errors="replace").splitlines()[-35:]
            print("\nLast log lines:", flush=True)
            print("\n".join(tail), flush=True)
            print("\nFull persistent log: " + str(LOG_PATH), flush=True)
        else:
            print(traceback.format_exc(), flush=True)
    except Exception:
        print(traceback.format_exc(), flush=True)
    raise
